In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn import functional as F
from scipy.optimize import minimize

In [2]:
block_size = 32
n_embd     = 64
n_head     = 4
n_layer    = 4
dropout    = 0.0
device     = 'cpu'

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars      = sorted(list(set(text)))
vocab_size = len(chars)
stoi       = {ch: i for i, ch in enumerate(chars)}

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x); q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f   = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, x):
        x = self.blocks(x)
        x = self.ln_f(x)
        return self.lm_head(x)

model = BigramLanguageModel().to(device)
model.load_state_dict(torch.load('model_weights.pth', map_location=device))
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
print(sum(p.numel() for p in model.parameters()) / 1e6, 'M parameters')

0.209729 M parameters


In [3]:
K, R = 4, 1.0
N, CHUNK = 50000, 2000
CONTEXT = text[int(0.9 * len(text)):][:block_size]
C_VECTOR = torch.zeros(vocab_size)
C_VECTOR[0], C_VECTOR[1] = 1.0, -1.0

tokens = torch.tensor([stoi[ch] for ch in CONTEXT])
x0 = (model.token_embedding_table(tokens) + model.position_embedding_table(torch.arange(len(tokens)))).detach()

lns = {name: m for name, m in model.named_modules() if isinstance(m, nn.LayerNorm)}
EPS = next(iter(lns.values())).eps
captured = {}
for name, m in lns.items():
    m.register_forward_pre_hook(lambda mod, inp, name=name: captured.__setitem__(name, inp[0][:, -1].detach()))

In [4]:
def run(z):
    x = x0.repeat(len(z), 1, 1)
    x[:, -1, :K] = x[:, -1, :K] + z
    return (model(x)[:, -1] @ C_VECTOR).sum()

with torch.no_grad():
    run(torch.zeros(1, K))
sigma0 = {name: captured[name].std(-1, unbiased=False).item() for name in lns}

torch.manual_seed(0)
sigmas, grads = {name: [] for name in lns}, []
for _ in range(N // CHUNK):
    z = ((2 * torch.rand(CHUNK, K) - 1) * R).requires_grad_(True)
    grads.append(torch.autograd.grad(run(z), z)[0].abs().sum(1))
    for name in lns:
        sigmas[name].append(captured[name].std(-1, unbiased=False))
grads = torch.cat(grads)
sigmas = {name: torch.cat(s) for name, s in sigmas.items()}
best = grads.argmax().item()
print(f'max sampled gradient norm: {grads.max():.4f}   sqrt(eps) = {EPS ** 0.5:.4g}')

max sampled gradient norm: 0.5275   sqrt(eps) = 0.003162


In [5]:
table = pd.DataFrame({
    'sigma at z=0': sigma0,
    'min sigma': {n: s.min().item() for n, s in sigmas.items()},
    'median sigma': {n: s.median().item() for n, s in sigmas.items()},
    'sigma at argmax': {n: s[best].item() for n, s in sigmas.items()},
})
table['min sigma / sqrt(eps)'] = table['min sigma'] / EPS ** 0.5
table['max gamma'] = {n: m.weight.abs().max().item() for n, m in lns.items()}
table['gain bound at min sigma'] = table['max gamma'] / np.sqrt(table['min sigma'] ** 2 + EPS)
table['gain bound at sigma=0'] = table['max gamma'] / EPS ** 0.5
table.round(4)

,sigma at z=0,min sigma,median sigma,sigma at argmax,min sigma / sqrt(eps),max gamma,gain bound at min sigma,gain bound at sigma=0
blocks.0.ln1,1.2465,1.2330,1.2539,1.2734,389.9153,1.1025,0.8941,348.6326
blocks.0.ln2,1.3778,1.3616,1.3836,1.4040,430.5717,0.9723,0.7141,307.4560
blocks.1.ln1,1.3491,1.3287,1.3549,1.3768,420.1755,1.3313,1.0019,420.9861
blocks.1.ln2,1.7655,1.7522,1.7693,1.7862,554.0952,1.1722,0.6690,370.6802
blocks.2.ln1,2.0947,2.0805,2.0943,2.1006,657.9056,1.2180,0.5854,385.1701
blocks.2.ln2,2.1973,2.1542,2.1962,2.2063,681.2262,1.3570,0.6299,429.1319
blocks.3.ln1,2.0920,2.0507,2.0903,2.0580,648.4779,1.3436,0.6552,424.8805
blocks.3.ln2,2.1088,2.0196,2.1064,2.0574,638.6585,1.4128,0.6996,446.7808
ln_f,2.0723,1.9539,2.0690,2.0300,617.8837,1.6651,0.8522,526.5554


In [6]:
a = x0[-1].numpy().astype(np.float64)
def var(z):
    v = a.copy()
    v[:K] += z
    return v.var()

res = minimize(var, np.zeros(K), bounds=[(-R, R)] * K)
print(f'blocks.0.ln1: certified min sigma over the box = {np.sqrt(res.fun):.4f}  (sigma at z=0: {np.sqrt(var(np.zeros(K))):.4f})')

blocks.0.ln1: certified min sigma over the box = 1.2330  (sigma at z=0: 1.2465)
